# Stage 2: ConvNeXt profiling

How long might full ConvNeXt-base fine-tuning take on the available GPU? This short compute probe uses random weights, so its predictions and loss are not model results. It is only a timing check for the planned pretrained run.

The existing reference is frozen ConvNeXt embeddings plus logistic regression: local macro F1 about 0.957 in the threshold-selection check, public leaderboard 0.92922. Neither score is measured here.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
import time
from pathlib import Path
from zipfile import ZipFile

import timm
import torch

KAGGLE_INPUT = Path("/kaggle/input")
PROGRESS_FILE = Path("/kaggle/working/convnext-profile-progress.jsonl") if KAGGLE_INPUT.is_dir() else None
started = time.perf_counter()

def mark(step: str, **details: object) -> None:
    line = json.dumps({"step": step, "elapsed_seconds": round(time.perf_counter() - started, 2), **details})
    print(line, flush=True)
    if PROGRESS_FILE is not None:
        with PROGRESS_FILE.open("a") as output:
            output.write(line + "\n")
            output.flush()
            os.fsync(output.fileno())

mark("setup_start")
if KAGGLE_INPUT.is_dir():
    dataset_roots = (
        KAGGLE_INPUT / "datasets" / "hermanhavva" / "ai301-stage2-convnext-profile-bundle",
        KAGGLE_INPUT / "datasets" / "ai301-stage2-convnext-profile-bundle",
        KAGGLE_INPUT / "ai301-stage2-convnext-profile-bundle",
    )
    dataset_root = next((path for path in dataset_roots if path.is_dir()), None)
    if dataset_root is None:
        raise FileNotFoundError("Profile bundle input not mounted at an expected path.")
    mark("bundle_root_ready", path=str(dataset_root))
    project_dirs = [path.parent.parent for path in dataset_root.rglob("stage2/profile.py")
                    if (path.parent.parent / "folds.csv").is_file()]
    if len(project_dirs) == 1:
        project_dir = project_dirs[0]
    else:
        bundles = list(dataset_root.rglob("stage2-profile-bundle.zip"))
        if len(bundles) != 1:
            raise FileNotFoundError("Attach exactly one Stage 2 profile bundle as a Kaggle input.")
        project_dir = Path("/kaggle/working/stage2-profile-bundle")
        with ZipFile(bundles[0]) as bundle:
            expected = {"folds.csv", "stage2/__init__.py", "stage2/profile.py"}
            if set(bundle.namelist()) != expected:
                raise ValueError("The attached profile bundle has unexpected files. Rebuild it from this repo.")
            bundle.extractall(project_dir)
    mark("bundle_ready", path=str(project_dir))
    data_dir = KAGGLE_INPUT / "competitions" / "cats-and-dogs"
    if not ((data_dir / "train.csv").is_file() and (data_dir / "test.csv").is_file()
            and (data_dir / "train").is_dir()):
        raise FileNotFoundError(f"Competition input not found at {data_dir}.")
    mark("data_root_ready", path=str(data_dir))
else:
    start = Path.cwd().resolve()
    project_dir = next(
        (path for path in (start, *start.parents)
         if (path / "folds.csv").is_file() and (path / "stage2" / "profile.py").is_file()),
        None,
    )
    if project_dir is None:
        raise FileNotFoundError("Launch Jupyter from the repository or its notebooks directory.")
    data_dir = project_dir / "data"
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
from stage2.profile import LABEL_COLUMNS, estimate_hours, load_sample, load_split, profile_updates
folds_csv = project_dir / "folds.csv"

MODEL_NAME = "convnext_base.fb_in22k_ft_in1k_384"
SEED = 42
VALIDATION_FOLD = 0
IMAGE_SIZE = 384
EFFECTIVE_BATCH_SIZE = 16
MICROBATCH_SIZES = (2, 4)  # Stop if 4 triggers an MPS memory error.
EPOCHS_PER_FOLD = 10
TOTAL_BUDGET_HOURS = 20
random.seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    raise RuntimeError("This full-backbone probe needs MPS or CUDA; CPU timing would not answer the budget question.")
mark("setup_done", **{
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "Apple MPS",
    "python": platform.python_version(),
    "torch": torch.__version__, "timm": timm.__version__,
    "data_dir": str(data_dir),
})


## Plan

Use the saved folds and exact five-label order. Prepare real images with aspect-preserving 384-pixel padding. Time full-backbone AdamW updates at an effective batch of 16 and a separate validation pass. The estimate is a capacity check, since a short run cannot capture all logging, saving, or thermal overhead.


In [ ]:
training_rows, validation_rows = load_split(data_dir, folds_csv, VALIDATION_FOLD)
sample_rows = random.sample(training_rows, EFFECTIVE_BATCH_SIZE)
mark("split_ready", training_images=len(training_rows), validation_images=len(validation_rows), labels=LABEL_COLUMNS)


## Image preparation

The whole 450×375 photo stays visible. Padding uses the checkpoint's mean colour so normalized padding is near zero. This differs from the earlier no-resize embedding baseline and will be recorded in model comparisons.


In [ ]:
mark("model_create_start")
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABEL_COLUMNS))
mark("model_created")
data_config = timm.data.resolve_model_data_config(model)
mean, std = tuple(data_config["mean"]), tuple(data_config["std"])
sample_images, sample_targets, image_seconds = load_sample(data_dir, sample_rows, IMAGE_SIZE, mean, std)
assert sample_images.shape == (EFFECTIVE_BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
mark("images_ready", image_seconds=round(image_seconds, 4), shape=tuple(sample_images.shape), normalization=(mean, std))


## Short compute probe

Each measured update accumulates gradients over 16 images, then runs AdamW. The same loaded images are reused to isolate model compute from disk I/O. One warm-up update is excluded. A synchronized progress marker follows every microbatch, which adds a little profiling overhead. No checkpoint is read or written.


In [ ]:
profile_results = {}
for microbatch_size in MICROBATCH_SIZES:
    mark("configuration_start", microbatch_size=microbatch_size)
    def report(stage: str, update: int, batch: int, seconds: float) -> None:
        mark("microbatch_progress", microbatch_size=microbatch_size, stage=stage,
             update=update, batch=batch, update_seconds=round(seconds, 2))
    profile_results[microbatch_size] = profile_updates(
        model, sample_images, sample_targets, DEVICE, microbatch_size, on_progress=report
    )
    mark("configuration_done", microbatch_size=microbatch_size, **profile_results[microbatch_size])
profile_results


## Capacity estimate

Five folds times ten full fine-tuning epochs is a deliberately broad scenario. The estimate omits head-only training, tuning, test inference, GradCAM, checkpoint saves, and long-run thermal effects. Keep a substantial reserve before scheduling the actual experiments.


In [ ]:
estimates = []
for microbatch_size, timings in profile_results.items():
    hours = estimate_hours(
        len(training_rows), len(validation_rows), EFFECTIVE_BATCH_SIZE, image_seconds,
        timings, epochs_per_fold=EPOCHS_PER_FOLD,
    )
    estimates.append({
        "microbatch": microbatch_size,
        **{key: round(value, 2) for key, value in hours.items()},
        "under_20h_before_overheads": hours["five_fold_hours_before_overheads"] < TOTAL_BUDGET_HOURS,
    })
estimates


## Next step

Record the GPU type, timing, and any memory error. Compare Kaggle with the local M2 result in the Stage 2 plan. A Kaggle session has its own time limit, so future training runs will save resumable checkpoints. Accuracy, tuning, and the custom block come after the unchanged-architecture benchmark.
